In [ ]:
%load_ext autoreload
%autoreload 2

import anndata as ad
from sklearn.preprocessing import StandardScaler
import numpy as np

# Load the AnnData object
adata = ad.read_h5ad("./data/cell_cycle/rpe1_kinetics_processed.h5ad")
adata

In [ ]:
s_genes = [
    "MCM5", "PCNA", "TYMS", "FEN1", "MCM7", "MCM4", "RRM1", "UNG", "GINS2", "MCM6",
    "CDCA7", "DTL", "PRIM1", "UHRF1", "CENPU", "HELLS", "RFC2", "POLR1B", "NASP",
    "RAD51AP1", "GMNN", "WDR76", "SLBP", "CCNE2", "UBR7", "POLD3", "MSH2", "ATAD2",
    "RAD51", "RRM2", "CDC45", "CDC6", "EXO1", "TIPIN", "DSCC1", "BLM", "CASP8AP2",
    "USP1", "CLSPN", "POLA1", "CHAF1B", "MRPL36", "E2F8"
]

g2m_genes = [
    "HMGB2", "CDK1", "NUSAP1", "UBE2C", "BIRC5", "TPX2", "TOP2A", "NDC80", "CKS2", "NUF2",
    "CKS1B", "MKI67", "TMPO", "CENPF", "TACC3", "PIMREG", "SMC4", "CCNB2", "CKAP2L",
    "CKAP2", "AURKB", "BUB1", "KIF11", "ANP32E", "TUBB4B", "GTSE1", "KIF20B", "HJURP",
    "CDCA3", "JPT1", "CDC20", "TTK", "CDC25C", "KIF2C", "RANGAP1", "NCAPD2", "DLGAP5",
    "CDCA2", "CDCA8", "ECT2", "KIF23", "HMMR", "AURKA", "PSRC1", "ANLN", "LBR", "CKAP5",
    "CENPE", "CTCF", "NEK2", "G2E3", "GAS2L3", "CBX5", "CENPA"
]

# --- Step 1: Filter all genes globally ---

# Extract all genes (not just cell cycle subset)
X_all = adata.layers["X_total"].toarray()
V_all = adata.layers["velocity_T"].toarray()

# Expression filter
avg_expr = X_all.mean(axis=0)
expr_threshold = np.quantile(avg_expr, 0.2)
expr_mask = avg_expr > expr_threshold

# Velocity filter
nonzero_velocity_mask = (V_all != 0).any(axis=0)

# Global mask
global_mask = expr_mask & nonzero_velocity_mask

X_filtered = X_all[:, global_mask]
V_filtered = V_all[:, global_mask]
genes_filtered = adata.var_names[global_mask]

phase_numeric = [float(i) for i in list(adata.obs["Cell_cycle_relativePos"])]
phase = list(adata.obs["cell_cycle_phase"])

# --- Step 2: Transform ---
X_log1p = np.log1p(X_filtered)
X_log1p = X_log1p - X_log1p.mean(axis=0, keepdims=True)
V_std = V_filtered / V_filtered.std(axis=0, ddof=0)

# --- Step 3: Subset to cell cycle genes ---
cell_cycle_genes = s_genes + g2m_genes
genes_present = [g for g in cell_cycle_genes if g in genes_filtered]

# Indices of those within the filtered set
gene_indices = [np.where(genes_filtered == g)[0][0] for g in genes_present]

X_cc = X_log1p[:, gene_indices]
V_cc = V_std[:, gene_indices]
genes_cc = np.array(genes_present)

X_cc.shape, V_cc.shape, len(genes_cc)

In [ ]:
import os
import matplotlib.pyplot as plt
import flowmap
from flowmap import *

emb = VectorFieldEmbedder(X_cc, V_cc, dist_method="phase",
                          embed_kwargs={"n_neighbors":30,
                                        "min_dist":0.6},
                          dof=30,method="umap")
emb.fit_embedding(1)

fig = flowmap.plot.plot_velocity_stream(
    X_2d=emb.X_emb,
    spline=emb.spline_vf,
    scatter_color=phase_numeric,
    grid_density=1.0, 
    stream_density=0.7,
    scatter_size=200,
    scatter_alpha=0.1,
    figsize=(5, 4),
    aspect=1,
    grid_size=50,
    show_axes=False,
    cmap="viridis",
    show_colorbar=False
)

plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl

alphas = [0.0, 0.25, 0.5,
          0.75, 1.0, 2.0,
          5.0, 10.0, 100.0]

fig, axes = plt.subplots(
    3,
    3,
    figsize=(18, 18),
    constrained_layout=True,
)

axes = axes.ravel()

# ------------------------------------------------------------
# consistent quiver subsampling
# ------------------------------------------------------------

rng = np.random.default_rng(0)
idx = rng.choice(len(X_cc), size=min(500, len(X_cc)), replace=False)

# ------------------------------------------------------------
# colormap
# ------------------------------------------------------------

cmap = mpl.colormaps["viridis"]

norm = mpl.colors.Normalize(
    vmin=np.min(phase_numeric),
    vmax=np.max(phase_numeric),
)

# colors for ALL cells
all_colors = cmap(norm(phase_numeric))

# colors for quiver subset
subset_colors = all_colors[idx]

for ax, alpha in zip(axes, alphas):

    print(f"Running alpha={alpha}")

    emb = VectorFieldEmbedder(
        X_cc,
        V_cc,
        dist_method="phase",
        embed_kwargs={
            "n_neighbors": 30,
            "min_dist": 0.6,
        },
        alpha=alpha,
        dof=30,
        method="umap",
    )

    emb.fit_embedding(1)

    Xp = emb.X_emb[idx]
    Vp = emb.V_emb[idx]

    # ----------------------------------------
    # normalize vectors for plotting only
    # ----------------------------------------

    norms = np.linalg.norm(Vp, axis=1, keepdims=True)
    norms[norms == 0] = 1
    Vp = Vp / norms

    # ----------------------------------------
    # scatter
    # ----------------------------------------

    ax.scatter(
        emb.X_emb[:, 0],
        emb.X_emb[:, 1],
        c=all_colors,
        s=20,
        alpha=0.08,
        edgecolors="none",
    )

    # ----------------------------------------
    # quivers
    # ----------------------------------------

    ax.quiver(
        Xp[:, 0],
        Xp[:, 1],
        Vp[:, 0],
        Vp[:, 1],
        color=subset_colors,
        angles="xy",
        scale_units="xy",
        scale=2,
        width=0.003,
        alpha=0.95,
    )

    ax.set_title(
        rf"$\alpha={alpha}$",
        fontsize=24,
    )

    ax.set_xticks([])
    ax.set_yticks([])

    ax.set_aspect("equal")

    # remove borders/spines
    for spine in ax.spines.values():
        spine.set_visible(False)

plt.suptitle(
    "Effect of phase-distance weighting on FlowMap embeddings",
    fontsize=40,
)

plt.show()

In [ ]:
%load_ext autoreload
%autoreload 2

import anndata as ad
# adata = ad.read_h5ad("./data/larry/postprocessed.h5ad")
adata = ad.read_h5ad("./data/larry/larry_processed.h5ad")
adata

In [ ]:
import numpy as np
import pandas as pd
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import StandardScaler

# 1. Get the matrix
# scVelo and Scanpy usually run PCA on log-transformed spliced counts
X = adata.layers["spliced"].copy()

# 2. Subset to highly variable genes
hvg_mask = adata.var["highly_variable"].values
X = X[:, hvg_mask]

# 3. Log1p transform (Scanpy's default)
X = X.toarray() if hasattr(X, "toarray") else X
X = np.log1p(X)

# 4. Center and scale (Scanpy centers but doesn’t always scale variance)
scaler = StandardScaler(with_mean=True, with_std=True)
X = scaler.fit_transform(X)

marker_dict = {
    "Mast": ["Cma1", "Tph1", "Papss2", "Fcer1a", "Gzmb"],
    "Basophil": ["Hgf", "Ccl3", "Slpi"],
    "Eosinophil": ["Prg3", "Epx", "Prg2"],
    "Megakaryocyte": ["Ppbp", "Thbs1", "Pf4", "Timp3"],
    "Monocyte": ["Ms4a6d", "Fabp5", "Ctss", "Ms4a6c", "Tgfbi",
                 "Olfm1", "Csf1r", "Ccr2", "Klf4", "F13a1"],
    "Neutrophil": ["S100a9", "Itgb2l", "Elane", "Fcnb",
                   "Mpo", "Prtn3", "S100a6", "S100a8",
                   "Lcn2", "Lrg1"],
    "Lymphoid": ["Ighm", "Satb1", "Dntt", "Ctr9", "Jchain"],
    "migDC": ["H2-Eb1", "H2-Aa", "H2-Ab1", "H2-DMb1", "Ccr7"],
    "pDC": ["Siglech"],
    "Erythroid": ["Hbb-bs"],
    "cDC": ["Cst3", "Xcr1"]
}

booster_dict = {
    "Mast": 1.0,
    "Basophil": 1.0,
    "Eosinophil": 1.0,
    "Megakaryocyte": 1.0,
    "Monocyte": 1.0,      # no boost
    "Neutrophil": 1.0,
    "Lymphoid": 1.0,
    "migDC": 1.0,
    "pDC": 1.0,
    "Erythroid": 1.0,
    "cDC": 1.0
}

# Copy to avoid side-effects
X_boosted = X.copy()

# Loop over cell types in booster_dict
for celltype, genes in marker_dict.items():
    boost = booster_dict.get(celltype, 1.0)
    if boost == 1.0:
        continue
    
    # find indices of marker genes among HVGs
    idx = [i for i, g in enumerate(adata.var_names[hvg_mask]) if g in genes]
    if idx:
        print(f"Boosting {celltype} markers ({len(idx)} genes) by {boost}x")
        X_boosted[:, idx] *= boost

from scipy.sparse import issparse

# ---------- 1. Load full dataset ----------

from sklearn.preprocessing import StandardScaler

# --- extract and subset ---
V = adata.layers["velocity"]  # or "velocity_pyro"
V = V[:, hvg_mask]
V = V.toarray() if hasattr(V, "toarray") else V

# --- compute per-gene mean, safely ---
gene_means = np.nanmean(V, axis=0)

# replace genes where mean is NaN (all-NaN columns)
nan_gene_mask = np.isnan(gene_means)
gene_means[nan_gene_mask] = 0.0  # set missing genes to 0

# --- impute ---
inds = np.where(np.isnan(V))
V[inds] = np.take(gene_means, inds[1])

# --- sanity check ---
print(f"Total NaN after imputation: {np.isnan(V).sum()}")

# --- scale (variance = 1 per gene) ---
scaler_v = StandardScaler(with_mean=False, with_std=True)
V = scaler_v.fit_transform(V)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

# ============================================================
# EXACT color mapping from main figure
# ============================================================

labels = np.asarray(adata.obs["state_info"].values)

# preserve appearance order
uniq_in_order = list(dict.fromkeys(labels))

other = [
    lab for lab in uniq_in_order
    if lab != "Undifferentiated"
]

# ------------------------------------------------------------
# exact colors
# ------------------------------------------------------------

tab10 = plt.get_cmap("tab10")

ordered_labels = ["Undifferentiated"] + other

ordered_colors = ["#d3d3d3"] + [
    mcolors.to_hex(tab10(i % 10))
    for i in range(len(other))
]

# categorical integer encoding
label_to_int = {
    lab: i
    for i, lab in enumerate(ordered_labels)
}

cell_color_ids = np.array([
    label_to_int[lab]
    for lab in labels
])

# custom colormap
custom_cmap = mcolors.ListedColormap(ordered_colors)

# ============================================================
# alpha sweep
# ============================================================

alphas = [0.0, 0.25, 0.5,
          0.75, 1.0, 2.0,
          5.0, 10.0, 100.0]

embeddings = {}

# ============================================================
# embedding params
# ============================================================

umap_params = {
    "min_dist": 0.5,
    "n_neighbors": 30,
}

# ============================================================
# run embeddings
# ============================================================

for alpha in alphas:

    print("\n" + "=" * 80)
    print(f"Running alpha={alpha}")
    print("=" * 80)

    emb = VectorFieldEmbedder(
        X_boosted,
        V,
        method="umap",
        spline_type="thin_plate",
        dist_method="phase",
        alpha=alpha,
        dof=50,
        pca_components=30,
        knn_k=30,
        n_control_points=4000,
        n_spline_points=None,
        embed_kwargs=umap_params,
    )

    emb.fit_embedding(seed=123)

    embeddings[alpha] = emb

    # --------------------------------------------------------
    # preview figure
    # --------------------------------------------------------

    fig, ax = plt.subplots(
        figsize=(8, 8),
        constrained_layout=True,
    )

    flowmap.plot.plot_velocity_stream(
        X_2d=emb.X_emb,
        spline=emb.spline_vf,
        scatter_color=cell_color_ids,
        cmap=custom_cmap,
        grid_size=45,
        grid_density=1.4,
        stream_density=2.2,   # MUCH denser
        scatter_size=14,
        scatter_alpha=0.35,
        arrowsize=1.0,
        ax=ax,
        aspect=1,
        show_axes=False,
        show_colorbar=False,
        streamline_thickness=2.8,
        pad_frac=0.02,
    )

    ax.set_title(
        rf"$\alpha={alpha}$",
        fontsize=24,
    )

    plt.show()

In [ ]:
# ============================================================
# combined grid
# ============================================================

fig, axes = plt.subplots(
    3,
    3,
    figsize=(20, 20),
    constrained_layout=True,
)

axes = axes.ravel()

for ax, alpha in zip(axes, alphas):

    emb = embeddings[alpha]

    flowmap.plot.plot_velocity_stream(
        X_2d=emb.X_emb,
        spline=emb.spline_vf,
        scatter_color=cell_color_ids,
        cmap=custom_cmap,
        grid_size=45,
        grid_density=1.4,
        stream_density=2.0,
        scatter_size=8,
        scatter_alpha=0.25,
        arrowsize=0.9,
        ax=ax,
        aspect=1,
        show_axes=False,
        show_colorbar=False,
        streamline_thickness=2.5,
        pad_frac=0.02,
    )

    ax.set_title(
        rf"$\alpha={alpha}$",
        fontsize=24,
    )


plt.show()